# Instructions for HW9:

**Student version.** Use Python 3.11 and install `requirements.txt`. Complete the eight TODO blocks by replacing each `raise NotImplementedError(...)` with your code, then restart the kernel and run all cells from the repository root.

Treat all quotes as `USD-SOFR-OIS` par swap rates and use this convention throughout. Follow the lecture notation; quote DV01 is in USD/bp and signed hedge notionals are $10^6w$.

1. **Apply PCA:** Use sklearn on all supplied quotes through 2026-02-26; retain the fewest PCs explaining at least 99% of variance.
2. **Calibration and risk:** Run the provided calibration, swap pricing, and Jacobian quote-risk code; assemble the hedge risk matrix $H$.
3. **PC loadings and exposures:** Display $U_k$, calculate $g$ and $A$, and reconstruct the portfolio's quote risk.
4. **Unweighted least squares:** Calculate minimum-norm hedge weights and report each instrument's signed notional and fixed-leg direction.
5. **Weighted least squares:** Use the retained PC eigenvalues to calculate weighted least-squares hedge positions and report the same instrument quantities.

In [ ]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from sklearn.decomposition import PCA

repo_root = next(
    folder for folder in (Path.cwd(), *Path.cwd().parents)
    if (folder / "fixedincomelib").is_dir()
)
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

from fixedincomelib import *
print("Fixed Income Library is loaded.")

plt.rcParams.update({"figure.dpi": 120, "axes.spines.top": False, "axes.spines.right": False})

## 1. Apply PCA

Apply sklearn PCA to all quotes and observations through 2026-02-26; retain the fewest PCs explaining at least 99% of variance.
Report eigenvalues, individual and cumulative explained-variance ratios, and the selected number of PCs.

In [ ]:
value_date = "2026-02-26"
swap_convention = "USD-SOFR-OIS"

quotes = pd.read_excel(repo_root / "swap_data.xlsx")
quotes["Dates"] = pd.to_datetime(quotes["Dates"])
swap_rates = quotes.set_index("Dates").sort_index().astype(float) / 100.0

tenors = swap_rates.columns.tolist()
tenor_years = np.array([
    float(tenor[:-1]) / 12.0 if tenor.endswith("M") else float(tenor[:-1])
    for tenor in tenors
])

print("Swap Rate Data (decimal rates):")
display(swap_rates.head())

In [ ]:
# TODO 1: Prepare the PCA input DataFrame X and fit sklearn PCA.
# Define the fitted pca object and its scores_array.
raise NotImplementedError("TODO 1: prepare the PCA input and fit the model.")

eigenvalues = pca.explained_variance_
explained_ratio = pca.explained_variance_ratio_
variance_target = 0.99
# TODO 2: Calculate cumulative_ratio and select the smallest k reaching variance_target.
raise NotImplementedError("TODO 2: select the number of PCs.")
pc_names = [f"PC{i + 1}" for i in range(len(eigenvalues))]
selected_pc_names = pc_names[:k]

selected_eigenvalues = pca.explained_variance_[:k]
scores = pd.DataFrame(scores_array, index=X.index, columns=pc_names)
explained_variance = pd.DataFrame({
    "Eigenvalue (bp squared)": eigenvalues,
    "Explained variance": explained_ratio,
    "Cumulative explained variance": cumulative_ratio,
    "Selected": np.arange(len(eigenvalues)) < k,
}, index=pd.Index(pc_names, name="Component"))

print(f"Historical quotes: {len(swap_rates)} dates; PCA observations: {len(X)}.")
print(f"Selected PCs: {k}; cumulative explained variance: {cumulative_ratio[k - 1]:.4%}.")
if k > 1:
    print(f"The first {k - 1} PCs explain {cumulative_ratio[k - 2]:.4%}, below the 99% threshold.")
display(explained_variance.style.format({
    "Eigenvalue (bp squared)": "{:.6f}",
    "Explained variance": "{:.4%}",
    "Cumulative explained variance": "{:.4%}",
}))

positions = np.arange(1, len(eigenvalues) + 1)
fig, axes = plt.subplots(1, 2, figsize=(11, 3.6), constrained_layout=True)
axes[0].bar(positions, 100 * explained_ratio, color="#27658b")
axes[0].set(xlabel="Principal component", ylabel="Explained variance (%)",
            title="Individual explained variance", xticks=positions)
axes[1].plot(positions, 100 * cumulative_ratio, "o-", color="#27658b", markersize=4)
axes[1].axhline(100 * variance_target, color="#b85b37", linestyle="--", label="99% threshold")
axes[1].axvline(k, color="#666666", linestyle=":", label=f"Selected k = {k}")
axes[1].set(xlabel="Number of principal components", ylabel="Cumulative explained variance (%)",
            title="Select the smallest k reaching 99%", xticks=positions, ylim=(90, 100.5))
axes[1].legend(fontsize=8, loc="lower right")
plt.show()

## 2. Calibration and Calculate Risk from Yield Curve

Run the provided 2026-02-26 calibration, pricing, and Jacobian quote-risk code (USD/bp), keeping the funding spread at zero.
Use the supplied portfolio risk vector $d$ and assemble $H$ from the hedge quote risks.

In [ ]:
funding_index = "SOFR-1B-FLAT"
funding_convention = "SOFR-1B-FLAT-OVER-SOFR-1B-ZERO-SPREAD"
funding_label = "USD-SOFR-FUNDING"

curve_quotes = swap_rates.loc[pd.Timestamp(value_date)].to_frame(name="values")
funding_quotes = pd.DataFrame({"values": [0.0]}, index=["30Y"])
funding_table = pd.DataFrame(
    [
        ["Overnight Index Swap", swap_convention, funding_index],
        ["Spread Zero Rate", funding_convention, funding_index],
    ],
    columns=["DATA TYPE", "DATA CONVENTION", "FUNDING IDENTIFIER"],
)

data_collection = qfCreateDataCollection([
    qfCreateData1D("Overnight Index Swap", swap_convention, curve_quotes),
    qfCreateData1D("Spread Zero Rate", funding_convention, funding_quotes),
    qfCreateDataGeneric("Data Generic", funding_label, funding_table),
])

build_methods = qfCreateModelBuildMethodCollection([
    qfCreateBuildMethod("YIELD_CURVE_COMMON", {
        "TARGET": "USD",
        "FUNDING PARAMETERS": funding_label,
        "SOLVER": "BRENTQ",
    }),
    qfCreateBuildMethod("YIELD_CURVE_FUNDING", {
        "TARGET": funding_index,
        "SPREAD ZERO RATE": funding_convention,
    }),
    qfCreateBuildMethod("YIELD_CURVE_INDEX", {
        "TARGET": "SOFR-1B",
        "OVERNIGHT INDEX SWAP": swap_convention,
    }),
])

yield_curve = qfCreateModel(value_date, "YIELD_CURVE", data_collection, build_methods)
valuation_parameters = qfCreateValuationParametersCollection([
    qfCreateValuationParameters("FUNDING INDEX PARAMETER", {
        "FUNDING INDEX": funding_index,
    }),
])
print(f"Yield curve is ready for {value_date}.")

**Portfolio pricing and quote risk**

Forward swaps (spot-date offset × tenor): `1Y×5Y, 1Y×10Y, 5Y×5Y, 5Y×1Y, 5Y×10Y, 10Y×5Y, 10Y×10Y`.
Run the provided code to price each USD 1m receive-fixed swap at its forward par rate and obtain its quote DV01 and aggregate $d$.

In [ ]:
portfolio_tenors = [
    "1Yx5Y", "1Yx10Y", "5Yx5Y", "5Yx1Y",
    "5Yx10Y", "10Yx5Y", "10Yx10Y",
]
portfolio_notional = 1e6
portfolio_products = {}
portfolio_rows = []
portfolio_risk_rows = []

for axis in portfolio_tenors:
    # Obtain the forward par rate through the library API.
    trial_swap = qfCreateProductFromDataConvention(
        value_date, swap_convention, axis, 0.04,
        notional=portfolio_notional, pay_or_rec="receive",
    )
    par_rate = float(qfCreateValueReport(
        yield_curve, trial_swap, valuation_parameters, "parrateorspread",
    ))
    product = qfCreateProductFromDataConvention(
        value_date, swap_convention, axis, par_rate,
        notional=portfolio_notional, pay_or_rec="receive",
    )
    pv = dict(qfCreateValueReport(
        yield_curve, product, valuation_parameters, "pv",
    ))["USD"]
    risk_report = qfCreateValueReport(
        yield_curve, product, valuation_parameters, "firstorderrisk",
    ).display()
    portfolio_products[axis] = product
    ois_rows = risk_report.loc[
        (risk_report["DATA_TYPE"].str.upper() == "OVERNIGHT INDEX SWAP")
        & (risk_report["DATA_CONVENTION"] == swap_convention)
    ]
    risk_vector = ois_rows.set_index("AXIS1")["VALUES"].astype(float).reindex(tenors)
    if risk_vector.isna().any():
        raise ValueError(f"Missing OIS quote risk for {axis}.")

    portfolio_rows.append({
        "Swap": axis,
        "Effective date": product.effective_date.ISO(),
        "Termination date": product.termination_date.ISO(),
        "Notional (USD)": product.notional,
        "Par rate": par_rate,
        "Fixed coupon": product.fixed_rate,
        "PV (USD)": float(pv),
    })
    portfolio_risk_rows.append(risk_vector.to_numpy())

portfolio_summary = pd.DataFrame(portfolio_rows).set_index("Swap")
portfolio_risk = pd.DataFrame(portfolio_risk_rows, index=portfolio_tenors, columns=tenors)
portfolio_risk.index.name = "Swap"
d = portfolio_risk.loc[:, tenors].sum(axis=0).to_numpy(dtype=float)
risk_agg = pd.DataFrame([d], columns=tenors, index=pd.Index(["AGG_RISK"], name="Portfolio"))
portfolio_pv = portfolio_summary["PV (USD)"].sum()

print("Portfolio pricing:")
display(portfolio_summary.style.format({
    "Notional (USD)": "{:,.0f}", "Par rate": "{:.6%}",
    "Fixed coupon": "{:.6%}", "PV (USD)": "{:.3e}",
}))
print(f"Total portfolio PV (USD): {portfolio_pv:.6e}")
print("Individual quote DV01s (USD per bp):")
display(portfolio_risk.style.format("{:,.4f}"))
print("Aggregated quote DV01 (USD per bp):")
display(risk_agg.style.format("{:,.4f}"))

**Hedging instruments**

Use 13 spot swaps: `1Y, 2Y, 3Y, 4Y, 5Y, 6Y, 7Y, 8Y, 9Y, 10Y, 12Y, 15Y, 20Y`, each USD 1m receive-fixed at par.
Build $H$ with one column per hedge and all 15 quote tenors as rows, including 25Y and 30Y.

In [ ]:
hedge_tenors = [
    "1Y", "2Y", "3Y", "4Y", "5Y", "6Y", "7Y",
    "8Y", "9Y", "10Y", "12Y", "15Y", "20Y",
]
hedge_notional = 1e6
hedge_products = {}
hedge_rows = []
hedge_risk_rows = []

for axis in hedge_tenors:
    trial_swap = qfCreateProductFromDataConvention(
        value_date, swap_convention, axis, 0.04,
        notional=hedge_notional, pay_or_rec="receive",
    )
    par_rate = float(qfCreateValueReport(
        yield_curve, trial_swap, valuation_parameters, "parrateorspread",
    ))
    product = qfCreateProductFromDataConvention(
        value_date, swap_convention, axis, par_rate,
        notional=hedge_notional, pay_or_rec="receive",
    )
    pv = dict(qfCreateValueReport(
        yield_curve, product, valuation_parameters, "pv",
    ))["USD"]
    risk_report = qfCreateValueReport(
        yield_curve, product, valuation_parameters, "firstorderrisk",
    ).display()
    hedge_products[axis] = product
    ois_rows = risk_report.loc[
        (risk_report["DATA_TYPE"].str.upper() == "OVERNIGHT INDEX SWAP")
        & (risk_report["DATA_CONVENTION"] == swap_convention)
    ]
    risk_vector = ois_rows.set_index("AXIS1")["VALUES"].astype(float).reindex(tenors)
    if risk_vector.isna().any():
        raise ValueError(f"Missing OIS quote risk for {axis}.")

    hedge_rows.append({"Tenor": axis, "Par rate": par_rate, "PV (USD)": float(pv)})
    hedge_risk_rows.append(risk_vector.to_numpy())

hedge_summary = pd.DataFrame(hedge_rows).set_index("Tenor")
hedging = pd.DataFrame(hedge_risk_rows, index=hedge_tenors, columns=tenors)
hedging.index.name = "Hedge tenor"
# TODO 3: Build H from hedging, with quote tenors as rows and hedge instruments as columns.
raise NotImplementedError("TODO 3: construct the hedge quote-risk matrix.")

print("Reference hedge swaps: USD 1,000,000 each, receive fixed at par.")
display(hedge_summary.style.format({"Par rate": "{:.6%}", "PV (USD)": "{:.3e}"}))
print("Hedging instruments' quote DV01s (USD per bp):")
display(hedging.style.format("{:,.4f}"))
print(f"Portfolio risk shape: {d.shape}; hedge risk matrix H shape: {H.shape}.")

## 3. PC Loadings and Risk Exposures

Display and plot the retained PC loadings $U_k$ across the original swap tenors.
Calculate portfolio and hedge PC exposures, then reconstruct portfolio risk in quote space.

In [ ]:
# TODO 4: Extract U_k from the fitted pca, with one retained PC loading vector per column.
raise NotImplementedError("TODO 4: obtain the retained PC loadings.")

loadings = pd.DataFrame(
    U_k, index=pd.Index(tenors, name="Original quote tenor"), columns=selected_pc_names,
)
print("Selected PC loadings on the original swap-rate tenors:")
display(loadings.style.format("{:.6f}"))

fig, axes = plt.subplots(1, k, figsize=(4.3 * k, 3.6), constrained_layout=True, squeeze=False)
for j, ax in enumerate(axes.ravel()):
    ax.plot(tenor_years, U_k[:, j], "o-", color="#27658b", markersize=4)
    ax.axhline(0.0, color="#777777", linewidth=0.8)
    ax.set(xlabel="Tenor (years)", ylabel="Loading", xticks=[1, 5, 10, 20, 30],
           title=f"PC{j + 1}: {explained_ratio[j]:.2%}")
plt.show()

**Portfolio and hedge risks in PC space**

Calculate and report $g=U_k^Td$ and $A=U_k^TH$, keeping the original quote-tenor order aligned.

In [ ]:
# TODO 5: Calculate portfolio PC exposures g and hedge PC exposures A.
raise NotImplementedError("TODO 5: project quote risks into PC space.")

portfolio_pc_risk = pd.DataFrame({
    "Portfolio exposure g": g,
}, index=pd.Index(selected_pc_names, name="Component"))
hedge_pc_risk = pd.DataFrame(
    A.T,
    index=hedge_tenors,
    columns=selected_pc_names,
)

print("Portfolio PC risk g (USD per bp of unit PC):")
display(portfolio_pc_risk.style.format("{:,.6f}"))
print("Reference hedge instruments' PC risks (rows of A.T, USD per bp of unit PC):")
display(hedge_pc_risk.style.format("{:,.6f}"))

**Reconstruct the portfolio risk**

Tabulate $d$, its projection $\widehat d=U_kg$, and the residual $d_\perp=d-\widehat d$; plot $d$ and $\widehat d$.

In [ ]:
# TODO 6: Calculate the projected risk d_hat and projection residual d_perp.
raise NotImplementedError("TODO 6: reconstruct the portfolio risk.")

risk_reconstruction = pd.DataFrame({
    "Original quote DV01 d": d,
    "Projected risk d_hat": d_hat,
    "Projection residual d_perp": d_perp,
}, index=pd.Index(tenors, name="Tenor"))

print("Risk reconstruction before hedging (USD per bp):")
display(risk_reconstruction.style.format("{:,.4f}"))

fig, ax = plt.subplots(figsize=(9, 4), constrained_layout=True)
ax.plot(tenor_years, d, "o-", color="#27658b", label="Original quote DV01 d")
ax.plot(tenor_years, d_hat, "s--", color="#b85b37", label=f"Projected risk d_hat ({k} PCs)")
ax.axhline(0.0, color="#777777", linewidth=0.8)
ax.set(xlabel="Tenor (years)", ylabel="Quote DV01 (USD per bp)",
       title="Portfolio risk and its PCA projection", xticks=[1, 5, 10, 15, 20, 25, 30])
ax.legend()
plt.show()

## 4. PCA Unweighted Least Squares

Solve $\min_w\lVert Aw+g\rVert_2^2$ with `np.linalg.lstsq`, selecting the minimum-norm solution.
Report each hedge instrument's weight, signed notional, and fixed-leg direction.

In [ ]:
# TODO 7: Calculate the minimum-norm unweighted hedge positions w_unweighted.
raise NotImplementedError("TODO 7: solve the unweighted least-squares hedge.")

unweighted_hedge = pd.DataFrame({
    "Weight": w_unweighted,
    "Signed notional (USD)": hedge_notional * w_unweighted,
    "Fixed-leg direction": np.where(w_unweighted >= 0.0, "Receive fixed", "Pay fixed"),
}, index=pd.Index(hedge_tenors, name="Hedge tenor"))
display(unweighted_hedge.style.format({"Weight": "{:.9f}", "Signed notional (USD)": "{:,.2f}"}))

## 5. PCA Weighted Least Squares

Solve $\min_w\lVert\Lambda_k^{1/2}(Aw+g)\rVert_2^2$ using the retained PCA eigenvalues and minimum-norm least squares.
Report each hedge instrument's weight, signed notional, and fixed-leg direction.

In [ ]:
# TODO 8: Use the retained PCA eigenvalues to calculate the minimum-norm hedge positions w_weighted.
raise NotImplementedError("TODO 8: solve the weighted least-squares hedge.")

weighted_hedge = pd.DataFrame({
    "Weight": w_weighted,
    "Signed notional (USD)": hedge_notional * w_weighted,
    "Fixed-leg direction": np.where(w_weighted >= 0.0, "Receive fixed", "Pay fixed"),
}, index=pd.Index(hedge_tenors, name="Hedge tenor"))
display(weighted_hedge.style.format({"Weight": "{:.9f}", "Signed notional (USD)": "{:,.2f}"}))